In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data")

def print_tree(path, max_depth=3, max_items_per_dir=20, current_depth=0):
    path = Path(path)

    if current_depth > max_depth:
        return

    try:
        items = list(path.iterdir())
    except PermissionError:
        return

    items = sorted(items, key=lambda x: (x.is_file(), x.name))[:max_items_per_dir]

    indent = "  " * current_depth

    for item in items:
        print(f"{indent}{'📄' if item.is_file() else '📁'} {item.name}")

        if item.is_dir():
            print_tree(item, max_depth, max_items_per_dir, current_depth + 1)

print_tree(DATASET_DIR, max_depth=4, max_items_per_dir=15)

In [ ]:
from collections import Counter
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data")

ext_counter = Counter()

for file in DATASET_DIR.rglob("*"):
    if file.is_file():
        ext_counter[file.suffix.lower()] += 1

for ext, count in ext_counter.most_common():
    print(ext if ext else "(확장자 없음)", count)

In [ ]:
# import json
# from pathlib import Path

# DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data")

# json_files = list(DATASET_DIR.rglob("*.json"))

# print("JSON 파일 개수:", len(json_files))
# print("첫 번째 JSON 파일:", json_files[0])

# with open(json_files[0], "r", encoding="utf-8") as f:
#     data = json.load(f)

# print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])

In [ ]:
!pip install soundfile

In [ ]:
import soundfile as sf
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data")

wav_files = list(DATASET_DIR.rglob("*.wav"))

print("WAV 파일 개수:", len(wav_files))
print("첫 번째 WAV 파일:", wav_files[0])

info = sf.info(str(wav_files[0]))
print(info)

In [ ]:
# from pathlib import Path
# import json
# import random
# from collections import Counter
# from tqdm import tqdm

# # 네 데이터셋 최상위 폴더로 수정
# DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data/New_Sample (oldman_voice)")

# label_files = list(DATASET_DIR.rglob("*.json"))
# audio_files = list(DATASET_DIR.rglob("*.wav"))

# print("라벨 JSON 개수:", len(label_files))
# print("음성 WAV 개수:", len(audio_files))

# # 파일명만 기준으로 wav 경로 찾기 쉽게 만들기
# audio_map = {p.name: p for p in audio_files}

# pairs = []
# missing_audio = []
# error_files = []

# for label_path in tqdm(label_files, desc="라벨 JSON 처리 중"):
#     try:
#         with open(label_path, "r", encoding="utf-8") as f:
#             data = json.load(f)

#         utter_info = data.get("발화정보", {})
#         conv_info = data.get("대화정보", {})
#         speaker_info = data.get("녹음자정보", {})

#         file_name = utter_info.get("fileNm")
#         reference_text = utter_info.get("stt")

#         audio_path = audio_map.get(file_name)

#         if audio_path is None:
#             missing_audio.append((str(label_path), file_name))
#             continue

#         pairs.append({
#             "label_path": str(label_path),
#             "audio_path": str(audio_path),
#             "file_name": file_name,
#             "reference_text": reference_text,
#             "record_time": utter_info.get("recrdTime"),
#             "quality": utter_info.get("recrdQuality"),
#             "region": conv_info.get("cityCode"),
#             "topic": conv_info.get("convrsThema", "").strip(),
#             "gender": speaker_info.get("gender"),
#             "age": speaker_info.get("age"),
#         })

#     except Exception as e:
#         error_files.append((str(label_path), str(e)))

# print("매칭 성공:", len(pairs))
# print("음성 파일 못 찾음:", len(missing_audio))
# print("오류 파일:", len(error_files))

# if error_files:
#     print("오류 예시:", error_files[:3])

In [ ]:
import json
from pathlib import Path

# 이미 위에서 만들었다면 재사용 가능
# label_files = list(DATASET_DIR.rglob("*.json"))
# audio_files = list(DATASET_DIR.rglob("*.wav"))

print("라벨 파일 예시:", label_files[0])
print("음성 파일 예시:", audio_files[0])

with open(label_files[0], "r", encoding="utf-8") as f:
    data = json.load(f)

json_file_name = data["발화정보"]["fileNm"]
real_audio_name = audio_files[0].name

print("JSON fileNm:")
print(repr(json_file_name))

print("\n실제 wav 파일명:")
print(repr(real_audio_name))

In [ ]:
from pathlib import Path
import json
import unicodedata
from tqdm import tqdm

DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data/New_Sample (oldman_voice)")

label_files = list(DATASET_DIR.rglob("*.json"))
audio_files = list(DATASET_DIR.rglob("*.wav"))

print("라벨 JSON 개수:", len(label_files))
print("음성 WAV 개수:", len(audio_files))

# 파일명 유니코드 정규화해서 wav 경로 매핑
audio_map = {
    unicodedata.normalize("NFC", p.name): p
    for p in audio_files
}

pairs = []
missing_audio = []
error_files = []

for label_path in tqdm(label_files, desc="라벨 JSON 처리 중"):
    try:
        with open(label_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        utter_info = data.get("발화정보", {})
        conv_info = data.get("대화정보", {})
        speaker_info = data.get("녹음자정보", {})

        file_name = utter_info.get("fileNm")
        reference_text = utter_info.get("stt")

        if file_name is not None:
            file_name = unicodedata.normalize("NFC", file_name)

        audio_path = audio_map.get(file_name)

        if audio_path is None:
            missing_audio.append((str(label_path), file_name))
            continue

        pairs.append({
            "label_path": str(label_path),
            "audio_path": str(audio_path),
            "file_name": file_name,
            "reference_text": reference_text,
            "record_time": utter_info.get("recrdTime"),
            "quality": utter_info.get("recrdQuality"),
            "region": conv_info.get("cityCode"),
            "topic": conv_info.get("convrsThema", "").strip(),
            "gender": speaker_info.get("gender"),
            "age": speaker_info.get("age"),
        })

    except Exception as e:
        error_files.append((str(label_path), str(e)))

print("매칭 성공:", len(pairs))
print("음성 파일 못 찾음:", len(missing_audio))
print("오류 파일:", len(error_files))

if missing_audio:
    print("매칭 실패 예시:")
    for item in missing_audio[:5]:
        print(item)

if error_files:
    print("오류 예시:")
    print(error_files[:3])

In [ ]:
import json

with open("/content/pairs_oldman_voice.json", "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)

print("저장 완료:", len(pairs))

In [ ]:
import json

with open("/content/pairs_oldman_voice.json", "r", encoding="utf-8") as f:
    pairs = json.load(f)

print("불러온 매칭 데이터 수:", len(pairs))

In [ ]:
from collections import Counter, defaultdict
import random

region_counter = Counter(item["region"] for item in pairs)

print("지역별 개수")
for region, count in region_counter.most_common():
    print(region, count)

by_region = defaultdict(list)

for item in pairs:
    by_region[item["region"]].append(item)

sample_items = []

for region, items in by_region.items():
    n = min(3, len(items))
    sample_items.extend(random.sample(items, n))

print("\n샘플 개수:", len(sample_items))

for i, item in enumerate(sample_items, start=1):
    print("=" * 70)
    print("번호:", i)
    print("지역:", item["region"])
    print("나이/성별:", item["age"], item["gender"])
    print("주제:", item["topic"])
    print("정답 전사:", item["reference_text"])
    print("음성 경로:", item["audio_path"])

In [ ]:
from IPython.display import Audio, display

item = sample_items[0]

print("지역:", item["region"])
print("정답 전사:", item["reference_text"])
display(Audio(item["audio_path"]))

In [ ]:
# !pip install transformers accelerate librosa soundfile

In [ ]:
from transformers import pipeline
import torch

MODEL_NAME = "seastar105/whisper-medium-komixv2"

device = 0 if torch.cuda.is_available() else -1

asr = pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    device=device,
)

print("device:", device)

In [ ]:
# !pip install -U transformers accelerate librosa soundfile

In [ ]:
item = sample_items[0]

result = asr(
    item["audio_path"],
    generate_kwargs={
        "language": "ko",
        "task": "transcribe"
    }
)

print("파일:", item["file_name"])
print("지역:", item["region"])
print("나이/성별:", item["age"], item["gender"])
print("정답:", item["reference_text"])
print("STT :", result["text"])

In [ ]:
stt_results = []

for i, item in enumerate(sample_items, start=1):
    result = asr(
        item["audio_path"],
        generate_kwargs={
            "language": "ko",
            "task": "transcribe"
        }
    )

    stt_text = result["text"].strip()

    row = {
        **item,
        "model_stt_text": stt_text
    }

    stt_results.append(row)

    print("=" * 70)
    print("번호:", i)
    print("지역:", item["region"])
    print("나이/성별:", item["age"], item["gender"])
    print("주제:", item["topic"])
    print("정답:", item["reference_text"])
    print("STT :", stt_text)